In [111]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

In [112]:
df = pd.read_csv("Sleep_health_and_lifestyle_dataset (1).csv")

df.head().T

,0,1,2,3,4
Person ID,1,2,3,4,5
Gender,Male,Male,Male,Male,Male
Age,27,28,28,28,28
Occupation,Software Engineer,Doctor,Doctor,Sales Representative,Sales Representative
Sleep Duration,6.1,6.2,6.2,5.9,5.9
Quality of Sleep,6,6,6,4,4
Physical Activity Level,42,60,60,30,30
Stress Level,6,8,8,8,8
BMI Category,Overweight,Normal,Normal,Obese,Obese
Blood Pressure,126/83,125/80,125/80,140/90,140/90


In [113]:
print("Shape:", df.shape)

Shape: (374, 13)


In [76]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Sleep Health and Lifestyle Dataset - Profiling Report",
    explorative=True
)

profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:00<00:00, 9759.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [114]:
if "Person ID" in df.columns:
    df = df.drop(columns=["Person ID"])

In [115]:
bp_col = None
for col in df.columns:
    if "blood" in col.lower() and "pressure" in col.lower():
        bp_col = col
        break

In [116]:
print("Blood Pressure column:", bp_col)

if bp_col is not None:
    bp_split = df[bp_col].astype(str).str.split("/", expand=True)
    df["Systolic_BP"] = pd.to_numeric(bp_split[0], errors="coerce")
    df["Diastolic_BP"] = pd.to_numeric(bp_split[1], errors="coerce")
    df = df.drop(columns=[bp_col])

Blood Pressure column: Blood Pressure


In [117]:
df["Systolic_BP"] = df["Systolic_BP"].fillna(df["Systolic_BP"].median())
df["Diastolic_BP"] = df["Diastolic_BP"].fillna(df["Diastolic_BP"].median())


In [118]:
df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic_BP,Diastolic_BP
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,NaN,126,83
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
3,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90
4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90


In [119]:
X = df.drop(columns=["Sleep Disorder"])
y = df["Sleep Disorder"]

print("Target value counts:")
print(y.value_counts())

Target value counts:
Sleep Disorder
Sleep Apnea    78
Insomnia       77
Name: count, dtype: int64


In [120]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Encoded classes:", list(le.classes_))
print("Encoded target shape:", y_encoded.shape)

Encoded classes: ['Insomnia', 'Sleep Apnea', nan]
Encoded target shape: (374,)


In [121]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)
print("Test class distribution:", np.bincount(y_test))

Train size: (299, 12)
Test size : (75, 12)
Test class distribution: [15 16 44]


In [122]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols   = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns  :", numerical_cols)

Categorical columns: ['Gender', 'Occupation', 'BMI Category']
Numerical columns  : ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic_BP', 'Diastolic_BP']


In [123]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline

In [124]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [125]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Processed Train shape:", X_train_processed.shape)
print("Processed Test shape :", X_test_processed.shape)


Processed Train shape: (299, 25)
Processed Test shape : (75, 25)


ANN

In [126]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# safety seed (again, no harm)
tf.random.set_seed(42)

input_dim = X_train_processed.shape[1]
num_classes = len(np.unique(y_train))

model = Sequential([
    Dense(64, activation="relu", input_dim=input_dim),
    Dropout(0.3),

    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


C:\Users\Shuvo\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,843 (15.01 KB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 0 (0.00 B)

In [127]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_processed,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3389 - loss: 1.1604 - val_accuracy: 0.8000 - val_loss: 0.9643
Epoch 2/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6820 - loss: 0.9117 - val_accuracy: 0.7500 - val_loss: 0.7986
Epoch 3/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7573 - loss: 0.7588 - val_accuracy: 0.7667 - val_loss: 0.6639
Epoch 4/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7573 - loss: 0.6549 - val_accuracy: 0.9167 - val_loss: 0.5519
Epoch 5/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8201 - loss: 0.5612 - val_accuracy: 0.9000 - val_loss: 0.4719
Epoch 6/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8577 - loss: 0.5048 - val_accuracy: 0.9000 - val_loss: 0.4149
Epoch 7/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8703 - loss: 0.4774 - val_accuracy: 0.9000 - val_loss: 0.3827
Epoch 8/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8703 - loss: 0.4582 - val_accuracy: 0.9000 - 

In [128]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_ann = np.argmax(model.predict(X_test_processed), axis=1)

print("FINAL ANN Accuracy:",
      round(accuracy_score(y_test, y_pred_ann) * 100, 2), "%")

print(confusion_matrix(y_test, y_pred_ann))
print(classification_report(y_test, y_pred_ann))


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
FINAL ANN Accuracy: 90.67 %
[[14  1  0]
 [ 2 13  1]
 [ 1  2 41]]
              precision    recall  f1-score   support

           0       0.82      0.93      0.88        15
           1       0.81      0.81      0.81        16
           2       0.98      0.93      0.95        44

    accuracy                           0.91        75
   macro avg       0.87      0.89      0.88        75
weighted avg       0.91      0.91      0.91        75



In [129]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(np.unique(y_train)),
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train_processed, y_train)

xgb_probs = xgb.predict_proba(X_test_processed)


In [93]:
ann_probs = model.predict(X_test_processed)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


In [130]:
# weights: XGB stronger than ANN
final_probs = (3 * xgb_probs + 2 * ann_probs) / 5
final_pred = np.argmax(final_probs, axis=1)

print("ENSEMBLE Accuracy:",
      round(accuracy_score(y_test, final_pred) * 100, 2), "%")

print(confusion_matrix(y_test, final_pred))
print(classification_report(y_test, final_pred))


ENSEMBLE Accuracy: 93.33 %
[[14  1  0]
 [ 1 14  1]
 [ 1  1 42]]
              precision    recall  f1-score   support

           0       0.88      0.93      0.90        15
           1       0.88      0.88      0.88        16
           2       0.98      0.95      0.97        44

    accuracy                           0.93        75
   macro avg       0.91      0.92      0.91        75
weighted avg       0.93      0.93      0.93        75



In [131]:
final_probs = (4 * xgb_probs + 1 * ann_probs) / 5
final_pred = np.argmax(final_probs, axis=1)

print("ENSEMBLE Accuracy:",
      round(accuracy_score(y_test, final_pred) * 100, 2), "%")


ENSEMBLE Accuracy: 93.33 %


In [132]:
final_pred = []

for i in range(len(y_test)):
    # if XGB very confident, trust it
    if np.max(xgb_probs[i]) > 0.70:
        final_pred.append(np.argmax(xgb_probs[i]))
    else:
        
        avg_prob = (4 * xgb_probs[i] + 1 * ann_probs[i]) / 5
        final_pred.append(np.argmax(avg_prob))

final_pred = np.array(final_pred)

print("ENSEMBLE Accuracy:",
      round(accuracy_score(y_test, final_pred) * 100, 2), "%")


ENSEMBLE Accuracy: 93.33 %


In [133]:
ann_probs_smoothed = ann_probs ** 1.2

final_probs = (4 * xgb_probs + 1 * ann_probs_smoothed) / 5
final_pred = np.argmax(final_probs, axis=1)

print("ENSEMBLE Accuracy:",
      round(accuracy_score(y_test, final_pred) * 100, 2), "%")


ENSEMBLE Accuracy: 93.33 %


In [134]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    xgb,
    X_train_processed,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("CV Accuracy:", scores)
print("Mean CV Accuracy:", scores.mean() * 100)


CV Accuracy: [0.93333333 0.88333333 0.88333333 0.86666667 0.89830508]
Mean CV Accuracy: 89.29943502824858
